# Training 101 — one full LocalAgent session, end to end

This walks the whole pipeline once, at toy scale, on a single GPU (Colab T4 is enough) or even CPU.
Every command is the same one a real run uses; only the budgets differ. By the end you will have
pretrained, midtrained and posttrained a model from random initialisation, scored it, and read its
loss curves.

## The three stages, and why they are separate

| Stage | Question it answers | Data | Budget |
|---|---|---|---|
| `pretrain` | Can it model language at all? | open text only | largest |
| `midtrain` | Does it know what an agent does? | open agentic data the benchmarks do **not** score | middle |
| `posttrain` | Does it emit the exact call asked for? | the benchmarks' own train splits | smallest |

The line between midtrain and posttrain is one question: **does this corpus correspond to a suite
the harness scores?** Yes puts it in posttrain, no puts it in midtrain. They are disjoint by
construction, which is what makes midtrain a measurement rather than a rehearsal — it has no
access to the benchmarks, so anything it buys has to show up as transfer.

## The metric

**Mean step-success over ten public agent benchmarks** — the function name *and every argument*
correct. Not name-only, not parse rate. See `docs/BENCHMARKS.md` for the claim boundary on each
suite.

## 0. Setup

On Colab pick **Runtime → Change runtime type → T4 GPU**. CPU works too, just slower — the toy
budgets below are sized so the whole notebook finishes either way.

In [ ]:
import subprocess, sys, os, json
from pathlib import Path

REPO = Path("LocalAgent")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/SangbumChoi/LocalAgent.git", str(REPO)], check=False)
if REPO.exists():
    os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1. Pick a tier

The ladder runs from 10M to 700M parameters. Each config declares the `param_budget` it is held
to — there is deliberately no environment variable that can raise it, because a silently-raised
ceiling is how a "sub-100M" claim stops being true.

`la-10m` is the one to learn on: it trains in minutes and every later tier is the same shape with
more capacity.

In [ ]:
subprocess.run(["openlocalagent", "model-info", "configs/model/la-10m.yaml"], check=True)

# The whole ladder, for context. Note active vs total on the sparse arm - that pair is the
# architecture experiment, and quoting its total alone would miss the entire point.
from openlocalagent.model.config import ModelConfig
for tier in ("la-10m", "la-45m", "la-93m", "la-150m", "la-300m", "la-300m-moe", "la-700m"):
    cfg = ModelConfig.from_yaml(f"configs/model/{tier}.yaml")
    active = cfg.estimate_active_params()
    total = cfg.estimate_params()
    note = f"  ({active:,} active)" if active != total else ""
    print(f"{tier:14s} {total:>12,}{note}")

## 2. Build a toy corpus

Real pretraining reads `data/shards/pt-big` — FineWeb-Edu, Cosmopedia v2 and permissive Python,
460.7M tokens, every source openly licensed. That is too much to download here, so this builds a
tiny public-domain sample through the *same* filter → dedup → tokenize → pack path.

The packer writes a manifest carrying source token counts, license counts and a split-assignment
hash, so a corpus can always be checked against the one a result was produced on.

In [ ]:
subprocess.run([sys.executable, "scripts/build_corpus.py", "--sample",
                "--out", "data/shards/sample", "--seq-len", "128",
                "--rows-per-shard", "64", "--val-fraction", "0.1"], check=True)

manifest = json.loads(Path("data/shards/sample/manifest.json").read_text())
print("train tokens:", f"{manifest['train_tokens']:,}")
print("documents:   ", f"{manifest['total_documents']:,}")

## 3. Stage 1 — pretrain

Loss starts near `ln(vocab)` for random logits and falls from there. With a 256-symbol byte
vocabulary that is about **5.55**; with the 16K BPE tokenizer the real tiers use, about **9.70**.
If it does not fall in the first hundred steps, something is wrong with the data or the learning
rate — not with your patience.

Every stage appends one record per step to `<out_dir>/curve.jsonl` **while it trains**, so a live
run and a crashed one both have a curve.

In [ ]:
subprocess.run(["openlocalagent", "train", "pretrain",
                "configs/pretrain/pretrain-speedrun.yaml"], check=True)

## 4. Read the curve

`read_curve` returns typed `CurvePoint` records, not raw dicts — so the fields are visible from the
signature rather than discovered by printing.

In [ ]:
from openlocalagent.train.curve import read_curve

points = read_curve("runs/pretrain-smoke")
print(f"{len(points)} steps recorded")
for p in points[::max(1, len(points) // 8)]:
    val = f"  val {p.validation_loss:.3f}" if p.validation_loss is not None else ""
    print(f"  step {p.step:4d}  loss {p.loss:.3f}  lr {p.learning_rate:.2e}{val}")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3))
    plt.plot([p.step for p in points], [p.loss for p in points], linewidth=1.4)
    plt.xlabel("step"); plt.ylabel("loss"); plt.title("pretrain"); plt.grid(alpha=.3)
    plt.show()
except ImportError:
    pass

## 5. Stages 2 and 3 — midtrain and posttrain

At real scale these read different corpora, and the split is the point:

- **midtrain** — 62,891 rows of agentic data the harness does not score: distillation traces, free
  episodes, Toucan, r0b0t-tools, device control.
- **posttrain** — 199,383 rows that *are* the benchmarks' train splits, plus curated and synthetic
  sets built from them.

Both stages render `openai_full_catalog_v1`, the same prompt contract the eval suite renders.
Training under a different one scores the model on a prompt shape it never saw, which reads as a
capability failure rather than a configuration one — a mistake worth avoiding once rather than
debugging twice.

The toy configs below use synthetic agent data so the notebook stays self-contained.

In [ ]:
subprocess.run(["openlocalagent", "synth", "configs/data/agent_synth.yaml"], check=False)
subprocess.run(["openlocalagent", "train", "midtrain",
                "configs/midtrain/midtrain-speedrun.yaml"], check=False)
print("\nAt real scale this is:")
print("  openlocalagent train midtrain  configs/midtrain/la-93m.yaml")
print("  openlocalagent train posttrain configs/posttrain/la-93m.yaml")

## 6. Score it

`openlocalagent-eval-suite` runs one evaluation process for every model, so a 10M byte-level agent and
a sub-1B instruct model are scored on identical tasks with identical metrics. Each adapter renders
the task in its own native format; the scoring is shared.

```bash
openlocalagent-eval-suite --model openlocalagent:runs/posttrain-la-93m/latest.pt \
                      --out results/evalsuite-full/la-93m.json --device cuda --batch-size 32
```

Two habits keep the number meaningful, and both are enforced rather than trusted:

- **An unknown `--suites` name is a hard error.** It used to be a silent skip, and a run once
  exited `0` having written a receipt missing the suite it was asked for. Two results had to be
  discarded as unreproducible.
- **Supplemental suites stay out of the headline ten.** τ²-bench runs only when named, because
  adding an eleventh would change what "the ten-benchmark average" means and break comparability
  with every receipt already recorded.

In [ ]:
from openlocalagent.eval.suite import SUITES, SUPPLEMENTAL_SUITES, RENAME_VIEWS

print("headline ten:")
for name in SUITES:
    print("  ", name)
print("supplemental (only when named):", list(SUPPLEMENTAL_SUITES))
print("diagnostic views  (only when named):", list(RENAME_VIEWS))

## 7. What to read next

| Question | Document |
|---|---|
| Where does anything live, and what do I touch to add a tier / suite / source? | `docs/CONVENTIONS.md` |
| What is each stage for, and how do I resume one? | `docs/STAGES.md` |
| What is every corpus, how big, what licence, how is contamination handled? | `docs/DATASETS.md` |
| What are the ten suites and what do the metrics mean? | `docs/BENCHMARKS.md` |

### Running it for real

On a GPU box, one command runs a tier's whole spine and scores it:

```bash
scripts/train_queue.sh 0 pretrain:93m midtrain:93m posttrain:93m evalsuite:93m
```

The queue runs jobs back to back, stops at the first failure so a stage never trains from a
checkpoint the stage before it failed to finish, and takes a per-job lock so two queues cannot
write the same run directory. Every stage sets `runtime.resume`, so re-running after an
interruption continues rather than restarting.